# 6장. 데이터를 보며 질문을 만드는 EDA

## 제출 정보
- 이름: 유은송
- GitHub ID: song-03
- 작성일: 2026.09.
- 최종 Notebook URL: 이미지 항목 좀 이상함

### 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


### 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


### 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


### 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [34]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: c:\dev\llm-data-analysis-course
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\reports


### 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [35]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


### 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [36]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [37]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


### 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [38]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| completed 주문 월별 매출 변화가 주문수, 평균주문금액 중 어떤 항목과 함께 변하는가?  | 날짜 변환 성공한 completed 주문(2025.07.09-2026.07.05.) |월별 `total_sales`, 고유 `order_id` 수, `avg_order_value` | `order_date` 변환 실패 수와 월 범위를 확인하고 월별 합계를 completed 원본 합계와 대조한다. |
| 고객별 구매금액 높은 사례는 반복 구매 혹은 고액 주문 중 어떤 형태로 나타나는가? | `orders`·`order_items`·`customers`를 연결한 completed 상세 | 고객별 고유 `order_count`, `total_sales`, `avg_order_value` | 고객 left join의 전후 행 수, 매칭을 확인하고, 고객별 합계를 completed 원본 합계와 대조한다. |
| completed 주문에서 카테고리별 판매 수량과 매출 기여도는 어떻게 다른가? | `order_items`를 `orders`·`products`와 연결한 completed 상세 474행 | `total_quantity`, `total_sales`, `sales_ratio`; `line_total` 합계 | `order_id`·`product_id` left join의 전후 행 수, 매칭을 확인하고, 카테고리 합계를 completed 원본 합계와 대조한다. 

### 내가 추가하거나 수정한 EDA 질문

- 질문: 고객별이 구매금액 높은 사례는 반복 구매 혹은 고액 주문 중 어떤 형태로 나타나는가?
- 이 질문을 선택한 이유: 분석 시 추후 우수 고객을 어떤 기준으로 선정할지에 대해 도움이 될 수 있고, 어떤 고객을 잡아야 하는지 마케팅에서 활용할 수 있다. 
- 필요한 데이터: `customers.csv`, `orders.csv`, `order_items.csv`
- 사용할 지표: 고객별 고유 `order_count`, `total_sales`, `avg_order_value`
- 현재 데이터로 답할 수 있다고 판단한 이유: cusotmer_id를 이용해 customers.csv와 orders.csv, order_items.csv를 연결해 각 고객이 어떤 상품을 몇개나 샀는지, 주문 금액이 얼마나 되는지 파악할 수 있기 떄문이다. 


### 왜 이 질문을 선택했는가?

위의 3가지 질문은 고객, 시간, 상품카테고리 관점에서 같은 completed 주문 상세정보를 보기 때문에 금액 기준을 일관적으로 비교할 수 있다. 데이터에는 주문일, 주문 상태, 상세 수량과 금액, 상품 카테고리, 고객 도시가 포함되어 있어 위 지표를 직접 계산할 수 있다. 
각각의 질문은 분석 후 추후 마케팅 등에 활용할 수도 있다. 첫번째 주문의 경우 월별 매출을 끌어올리기 위해 주문 수량을 올려야 하는지, 평균 주문 금액을 올려야 하는지 어떤 방향으로 마케팅을 진행할지 결정하는 것에 도움이 될 수 있다. 두번쨰 질문의 경우 우수 고객을 어떤 기준으로 선정할지에 대해 도움이 될 수 있으며, 세번째 질문의 경우 판매 수량과 매출 기여도를 분리해서 봄으로써 매출 분석에 도움이 되는 지표를 만들 수 있다.
프로모션이나 고객 선호처럼 결과의 원인을 설명할 수 있는 정보는 현재 데이터에 없기 때문에 각 질문은 원인을 추정하기보다 데이터에서 확인할 수 있는 분포와 변수 간 관계를 살펴보는 범위로 한정했다.

### 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [39]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [40]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [41]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,female,84,56.0
1,male,66,44.0


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


### 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [42]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [43]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [44]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


### 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [45]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [46]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [47]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00
주문 월 개수: 13


### 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [48]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [67]:
all_sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', all_sales_items.shape)
print('상품명 누락:', all_sales_items['product_name'].isna().sum())
print('카테고리 누락:', all_sales_items['category'].isna().sum())

all_sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


다만 all_sales_itmes는 order status를 반영하지 않았으므로 completed 주문 건만을 확인하기 위한 코드를 추가하였다.

In [68]:
order_sales_all = order_items.merge(
    orders,
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator='_order_merge',
)

print('병합 전 order_items 행 수:', len(order_items))
print('병합 후 order_sales_all 행 수:', len(order_sales_all))
print('orders 미매칭 행 수:', (order_sales_all['_order_merge'] != 'both').sum())

completed_order_sales = order_sales_all.loc[
    order_sales_all['order_status'].eq('completed')
].copy()
print('completed 상세 행 수:', len(completed_order_sales))
print('completed 고유 주문 수:', completed_order_sales['order_id'].nunique())
print('completed 원본 line_total 합계:', completed_order_sales['line_total'].sum())

sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator='_product_merge',
)
print('상품 병합 후 행 수:', len(sales_items))
print('상품 미매칭 행 수:', (sales_items['_product_merge'] != 'both').sum())
print('카테고리 누락 행 수:', sales_items['category'].isna().sum())


병합 전 order_items 행 수: 764
병합 후 order_sales_all 행 수: 764
orders 미매칭 행 수: 0
completed 상세 행 수: 474
completed 고유 주문 수: 184
completed 원본 line_total 합계: 148990000
상품 병합 후 행 수: 474
상품 미매칭 행 수: 0
카테고리 누락 행 수: 0


### 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [69]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


In [52]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,468,50174000,19.63,111579.0
1,뷰티,376,47551000,18.60,117688.0
2,전자기기,401,41003000,16.04,101588.0
3,생활용품,390,34839000,13.63,96438.0
4,식품,240,33597000,13.14,137143.0
5,도서,238,24645000,9.64,106857.0
6,패션,220,23801000,9.31,115909.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


### 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [70]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-05-07,card,completed,2026-05,Thursday
1,2,1,87,5,25000,125000,123,2026-05-07,card,completed,2026-05,Thursday
2,3,1,7,3,142000,426000,123,2026-05-07,card,completed,2026-05,Thursday
3,4,1,9,3,193000,579000,123,2026-05-07,card,completed,2026-05,Thursday
4,5,2,72,4,189000,756000,77,2025-07-23,naver_pay,cancelled,2025-07,Wednesday


In [73]:
#앞서 수정한 코드 반영한 코드
order_sales = completed_order_sales.copy()
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('completed 상세 행 수:', len(order_sales))
print('order_date 누락 행 수:', order_sales['order_date'].isna().sum())
print('completed 주문 시작일:', order_sales['order_date'].min())
print('completed 주문 종료일:', order_sales['order_date'].max())


completed 상세 행 수: 474
order_date 누락 행 수: 0
completed 주문 시작일: 2025-07-09 00:00:00
completed 주문 종료일: 2026-07-05 00:00:00


In [74]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


### 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [72]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


In [76]:
#앞선 코드 반영한 코드
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
    validate='many_to_one',
    indicator='_customer_merge',
)
print('고객 병합 전 행 수:', len(order_sales))
print('고객 병합 후 행 수:', len(customer_sales_base))
print('고객 미매칭 행 수:', (customer_sales_base['_customer_merge'] != 'both').sum())

customer_sales = (
    customer_sales_base
    .groupby(['customer_id', 'city'], dropna=False, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)
print(customer_sales.head(10))

source_total = completed_order_sales['line_total'].sum()
category_total = category_sales['total_sales'].sum()
monthly_total = monthly_sales['total_sales'].sum()
customer_total = customer_sales['total_sales'].sum()
print()
print('집계 총합 검증')
print('completed 원본 합계:', source_total)
print('카테고리별 합계:', category_total)
print('월별 합계:', monthly_total)
print('고객별 합계:', customer_total)
print('일치 여부:', 'O' if source_total == category_total == monthly_total == customer_total else 'CHECK')


고객 병합 전 행 수: 474
고객 병합 후 행 수: 474
고객 미매칭 행 수: 0
    customer_id city  order_count  total_sales  avg_order_value
76          117   성남            5      4100000         820000.0
62          102   고양            4      3996000         999000.0
51           83   수원            4      3880000         970000.0
21           30   서울            5      3590000         718000.0
29           40   서울            4      3523000         880750.0
13           20   인천            2      3191000        1595500.0
0             3   성남            2      3178000        1589000.0
70          111   광주            3      3153000        1051000.0
42           66   서울            4      3093000         773250.0
97          147   부산            2      2990000        1495000.0

집계 총합 검증
completed 원본 합계: 148990000
카테고리별 합계: 148990000
월별 합계: 148990000
고객별 합계: 148990000
일치 여부: O


## 2. 핵심 집계 결과
- completed 원본 합계: 148990000원
- 카테고리별 합계: 148990000원
- 월별 합계: 148990000원
- 총합 일치 여부: PASS 
(금액 분석에서는 전체 주문을 사용하지 않고 completed 주문만 포함했다. 이에 연결된 주문 상세 474행을 기준으로 분석했다.)

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
order_items와 orders를 병합한 결과, 병합 전후 모두 764행으로 행 수가 같았고 orders와 연결되지 않은 행도 없었다. completed 주문의 line_total 합계는 148,990,000원이었다. 상품 병합 후에도 상품과 카테고리 미매칭은 모두 0행이었다.
completed 주문 분포를 확인한 결과, 고객 150명 중 성남이 21명(14.00%)으로 가장 많았다. 성별은 F 84명(56.0%), M 66명(44.0%)이었다. 전체 주문 300건 중 completed는 184건(61.33%)이었다. 카테고리별로는 스포츠가 31743000원(21.31%)으로 가장 컸고, 수량도 295개로 가장 많았다. 반면 패션은 10587000원(7.11%), 수량 111개로 가장 작았다. 월별로는 2025-10이 25766000원, 26건으로 가장 높았다. 2026-06은 2826000원, 4건이었다. 

### 나의 해석과 판단
스포츠의 매출 비중이 가장 컸으나 이를 바로 고객 선호로 해석하기에는 이르다. 스포츠는 판매 수량도 가장 많고 평균 상품 가격도 111579원이었기 때문에 판매 수량과 상품 가격이 함께 영향을 준 것으로 판단했다. 
2025년 10월이 월매출이 높았는데, 이 역시 26건의 주문 수와 주문당 평균 991000원이 함께 높게 나타난 결과라고 판단했다. 다음 분석에서는 월별 매출과 카테고리별 매출을 함께 비교해 특정 시기나 카테고리에 매출이 집중되는지 확인할 필요가 있다.

### 업무·분석적 의미
completed 주문을 기준으로 검증한 결과 카테고리별, 원본, 월별 합계의 총금액이 모두 일치했다. 따라서 이후 카테고리, 월 단위 분석에서도 같은 매출 기준을 사용할 수 있다. 따라서 본격적인 분석 전 이를 검증하는 것은 추후 업무, 분석에서 데이터의 신뢰도를 위해 필요한 과정이다. 

### 한계와 추가 확인 사항
line_total은 completed 주문에 포함된 주문 상세 금액의 합계이다. 할인, 배송비, 세금, 부분 환불, 정산 시점은 반영하지 않았으므로 회계상 순매출과 같다고 볼 수는 없다. 또한 첫 달과 마지막 달은 완전한 한 달이 아니어서 다른 월과 단순 비교할 때 주의가 필요하다. 그리고 매출 차이가 발생한 원인까지 현재 데이터로 확인하기는 어렵다. 이후 월별 카테고리 구성과 주문당 품목 수를 추가로 확인하고, 할인이나 캠페인 정보가 있다면 함께 살펴볼 필요가 있다.



## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 스포츠는 85건의 주문에서 295개가 판매됐고, 패션은 33건의 주문에서 111개가 판매됐다.
- 가설: 스포츠의 높은 판매 수량은 주문 1건당 많이 담겼기보다, 스포츠를 포함한 주문 자체가 더 많아서 나타났을 수 있다.
- 추가 검증: 스포츠와 패션의 주문 수, 총판매 수량, 주문당 평균 판매 수량을 비교한다.
- 검증 결과: 주문당 평균 판매 수량은 스포츠 3.47개, 패션 3.36개로 유사했지만 주문 수는 스포츠가 85건으로 패션 33건보다 많았으므로 스포츠의 판매량 우위는 주로 주문 수 차이에서 나타났다고 볼 수 있다.

### 사례 2
- 관찰: 2025년 10월은 매출 25766000원, 주문 26건, 평균 주문금액 991000원으로 가장 높았다. 2026년 6월은 매출 2826000원, 주문 4건, 평균 주문금액 706500원이었다. 두 달의 매출 차이는 22940000원이었다.
- 가설: 매출 차이에 주문수 차이가 영향을 주었을 수 있다.
- 추가 검증: 주문수, 주문당 금액/수량/카테고리 구성을 비교하고 주문수와 주문당 금액 차이를 매출 차이로 분해한다.
- 검증 결과: 주문 수 차이의 효과는 15543000원, 주문당 금액 차이 효과는 7397000원으로 주문 수 효과가 더 컸다. 

### 사례 3
- 관찰: 익명 고객 ID 117은 총구매금액 4100000원으로 가장 높았고, 주문 수는 5회, 평균 주문금액은 820000원이었다. 다음으로 총구매금액이 높은 ID 102는 총구매금액 3996000원, 주문 수 4회, 평균 주문금액 999000원이었다.
- 가설: 높은 총구매금액에는 주문 건수가 크게 작용할 수도 있다.
- 추가 검증: 고객별 `order_count`와 `avg_order_value`의 분포·사분위수를 계산하고, 두 유형의 카테고리 구성 및 주문 시점을 비교한다.
- 검증 결과: ID 117(5회)과 ID 102(4회)은 전체 고객의 주문 횟수 중앙값(2회)보다 많이 주문했으며, 평균주문금액은 각각 820,000원과 999,000원으로 상위 25% 기준인 1,061,750원보다 낮아 높은 총구매금액에는 반복 구매의 영향이 더 크게 작용한 것으로 보인다.

In [81]:
# 사례 1

sports_fashion_check = (
    sales_items
    .query("category in ['스포츠', '패션']")
    .groupby('category', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
    )
)

sports_fashion_check['items_per_order'] = (
    sports_fashion_check['total_quantity']
    / sports_fashion_check['order_count']
).round(2)

sports_fashion_check = sports_fashion_check.sort_values(
    'order_count',
    ascending=False,
)

sports_fashion_check

,category,order_count,total_quantity,items_per_order
0,스포츠,85,295,3.47
1,패션,33,111,3.36


In [80]:
#사례 2
month_order = (
    order_sales
    .groupby(['order_month', 'order_id'], as_index=False)
    .agg(
        order_total=('line_total', 'sum'),
        item_quantity=('quantity', 'sum'),
    )
)

month_driver = (
    month_order
    .groupby('order_month', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('order_total', 'sum'),
        avg_order_value=('order_total', 'mean'),
        avg_items_per_order=('item_quantity', 'mean'),
    )
)

target_months = (
    month_driver
    .query("order_month in ['2025-10', '2026-06']")
    .sort_values('order_month')
    .copy()
)

october = target_months.loc[target_months['order_month'].eq('2025-10')].iloc[0]
june = target_months.loc[target_months['order_month'].eq('2026-06')].iloc[0]

# 2026-06을 기준으로 한 순차 분해: 매출차이 = 주문 수 효과 + 주문당 금액 효과
order_count_effect = (
    (october['order_count'] - june['order_count']) * june['avg_order_value']
)
aov_effect = october['order_count'] * (
    october['avg_order_value'] - june['avg_order_value']
)

comparison = pd.DataFrame({
    'metric': ['sales_difference', 'order_count_effect', 'avg_order_value_effect'],
    'value': [
        october['total_sales'] - june['total_sales'],
        order_count_effect,
        aov_effect,
    ],
}).round(0)

# 주문 월 정보를 카테고리별 상세에 연결
sales_items_month = (
    sales_items
    .drop(columns='order_month', errors='ignore')
    .merge(
        order_sales[['order_id', 'order_month']].drop_duplicates(),
        on='order_id',
        how='left',
        validate='many_to_one',
    )
)

monthly_category_compare = (
    sales_items_month
    .query("order_month in ['2025-10', '2026-06']")
    .groupby(['category', 'order_month'], as_index=False)
    .agg(total_sales=('line_total', 'sum'))
    .pivot(index='category', columns='order_month', values='total_sales')
    .fillna(0)
)

monthly_category_compare['difference_oct_minus_jun'] = (
    monthly_category_compare['2025-10'] - monthly_category_compare['2026-06']
)

monthly_category_compare = monthly_category_compare.sort_values(
    'difference_oct_minus_jun',
    ascending=False,
)

print('두 달 카테고리별 합계:', monthly_category_compare[['2025-10', '2026-06']].sum().sum())
print(
    '두 달 원본 합계:',
    order_sales.loc[
        order_sales['order_month'].isin(['2025-10', '2026-06']),
        'line_total'
    ].sum()
)

monthly_category_compare

두 달 카테고리별 합계: 28592000.0
두 달 원본 합계: 28592000


order_month,2025-10,2026-06,difference_oct_minus_jun
category,,,
스포츠,5799000.0,386000.0,5413000.0
생활용품,6027000.0,963000.0,5064000.0
뷰티,4606000.0,32000.0,4574000.0
전자기기,4118000.0,1445000.0,2673000.0
도서,2550000.0,0.0,2550000.0
식품,1899000.0,0.0,1899000.0
패션,767000.0,0.0,767000.0


In [79]:
#사례 3 코드
top_customer_ids = [117, 102]

# 전체 구매 고객 대비 위치 확인
customer_distribution = (
    customer_sales[['order_count', 'total_sales', 'avg_order_value']]
    .describe(percentiles=[.25, .5, .75, .9])
    .round(0)
)

top_customer_summary = (
    customer_sales
    .query('customer_id in @top_customer_ids')
    .sort_values('total_sales', ascending=False)
)

# 주문별 금액과 시점: 한 주문 안의 여러 상품 행을 주문 단위로 합산
top_customer_orders = (
    order_sales
    .query('customer_id in @top_customer_ids')
    .groupby(['customer_id', 'order_id', 'order_date'], as_index=False)
    .agg(
        order_total=('line_total', 'sum'),
        item_quantity=('quantity', 'sum'),
    )
    .sort_values(['customer_id', 'order_date'])
)

# 고객별 카테고리 구성과 매출 비중
top_customer_category = (
    sales_items
    .query('customer_id in @top_customer_ids')
    .groupby(['customer_id', 'category'], as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
)

top_customer_category['sales_ratio_within_customer'] = (
    top_customer_category['total_sales']
    / top_customer_category.groupby('customer_id')['total_sales'].transform('sum')
    * 100
).round(2)

print('고객별 매출 합계:', top_customer_summary['total_sales'].sum())
print('선택 고객 상세 매출 합계:', top_customer_orders['order_total'].sum())

customer_distribution, top_customer_summary, top_customer_orders, top_customer_category

고객별 매출 합계: 8096000
선택 고객 상세 매출 합계: 8096000


(       order_count  total_sales  avg_order_value
 count        100.0        100.0            100.0
 mean           2.0    1489900.0         825066.0
 std            1.0     958540.0         397371.0
 min            1.0     112000.0          56000.0
 25%            1.0     828750.0         528250.0
 50%            2.0    1296500.0         807000.0
 75%            2.0    2008500.0        1061750.0
 90%            3.0    2963000.0        1308800.0
 max            5.0    4100000.0        1931000.0,
     customer_id city  order_count  total_sales  avg_order_value
 76          117   성남            5      4100000         820000.0
 62          102   고양            4      3996000         999000.0,
    customer_id  order_id order_date  order_total  item_quantity
 1          102       177 2025-07-27       915000              9
 3          102       197 2025-08-10      1649000             13
 2          102       178 2025-10-14       140000              1
 0          102       139 2026-02-21      1

### 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [56]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


### 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [57]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_customer_city.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_product_category.csv')]

### 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [58]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

### 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

### 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

### 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

### 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

### 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

### 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

### 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\reports\ch06_eda_summary.md


### 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [59]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [60]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


### 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


### 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.


[사용 프롬프트]

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 한다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total
금액성 분석 기준:
- line_total = quantity × unit_price
- order_status == completed만 사용
현재까지 내가 직접 확인한 관찰:
- 관찰: 스포츠는 85건의 주문에서 295개가 판매됐고, 패션은 33건의 주문에서 111개가 판매됐다.
- 관찰: 2025년 10월은 매출 25766000원, 주문 26건, 평균 주문금액 991000원으로 가장 높았다. 2026년 6월은 매출 2826000원, 주문 4건, 평균 주문금액 706500원이었다. 두 달의 매출 차이는 22940000원이었다.


요청:
1. 현재 데이터로 추가 검증할 가치가 있는 EDA 질문 3개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋,pandas 기능, 사용할 지표, 왜 확인할 가치가 있는지, 현재 데이터로 답할 수 없는 부분 등을 함께 간략히 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
5. 코드는 작성하지 마세요, 데이터에 없는 원인을 만들지 마세요.최종 선택은 제가 함게 남겨두세요. 
```

**교수님께: ipynb 파일에는 질문 10개 만들라고 되어있는데, 실습가이드에는 정확히 3개만 제안하라고 해서 우선 3개만 제안하는 방안으로 진행했습니다*


## 4. LLM 질문 확장
- Safe Context: customers, products, orders, order_items의 컬럼 정보와 completed 주문만 금액 분석에 사용한다는 기준을 제공했다. 스포츠와 패션, 2025년 10월과 2026년 6월의 집계 결과도 함께 제공했다. 할인, 프로모션, 고객 선호처럼 데이터에서 확인할 수 없는 내용은 제외했다.
- Prompt: (20번의 내용과 동일) 데이터셋 구조, 금액성 분석 기준, 현재까지 내가 확인한 관찰을 제공, 추가 검증할 가치가 있는 EDA 질문 3개와 각 질문에 필요한 데이터셋, pandas 기능, 사용할 계산가능한 지표, 현재 데이터로 답할 수 없는 질문, 확인할 가치를 제안해달라고 요청.
- LLM 제안 요약:
    1. 스포츠와 패션의 매출 및 판매 수량 차이가 주문 수와 주문당 수량 중 어떤 지표와 함께 나타나는지 확인하는 질문을 제안
    2. 둘째, 2025년 10월과 2026년 6월의 매출 차이를 주문 수, 평균 주문금액으로 나누어 확인하도록 제안
    3. 구매금액 상위 고객이 반복 구매형인지 고액 주문형인지 확인하는 질문을 제안
- 실제 사용 여부: 사용
- 판단 이유: 실제로 내가 앞서 관찰-가설-추가검증에서 했던 애용과 동일한 내용이었다. 따라서 충분히 사용가능한 질문이라고 생각하였다. 또한 확인할 가치가 있는 이유 경우 내가 생각했던 내용을 더 깔끔하게 정리하여 주었다. 
- 사람이 수정한 내용: 딱히 없음. (이미 앞서 모두 검증한 사항이기 때문에 수정할 사항도 없었다.)

![LLM EDA 질문 확장](images/step04_llm.png)

### 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


### LLM 결과 해석
- Safe Context: 카테고리별 판매 수량, 총매출, 매출 비중만 제공했다. 할인, 프로모션, 고객 선호, 재구매 원인처럼 현재 데이터에서 직접 확인할 수 없는 내용은 제외했다. 입력 표의 카테고리별 수치는 해석 방식 예시이므로 현재 분석 데이터의 실제 집계값과 다를 수 있다는 점도 함께 제시했다.
- Prompt: 전자기기, 생활용품, 패션, 식품에 대한 총 수량, 총 매출, sales_ratio 제공하며 보고서에 넣을 수 있게 결과를 해석해달라고 요청했다..
- LLM 제안 요약: 판매 수량과 매출 비중이 다르게 나타날 수 있다는 점을 관찰로 제시했다. 단가 차이는 가설로 두고, 추가 분석으로 카테고리별 실제 판매 단가와 주문당 금액 비교, 반복 구매 패턴 확인을 제안했다.
- 실제 사용 여부: 수정 후 사용 
- 판단 이유: 관찰, 가설, 추가 분석을 구분한 형식은 보고서에 활용할 수 있었다. 하지만 단가와 재구매 원인은 현재 표만으로 확인할 수 없어 가설이나 추가 분석 항목으로만 남겨야 한다고 생각했다. 재구매율은 기준이 모호해 고객별 주문 횟수와 카테고리별 재구매 고객 수를 직접 계산하는 방식으로 바꿨다.
    - ‘재구매율 분석’은 별도의 재구매율 기준이 정해져 있지 않아 제외. 대신 주문 수, 총판매 수량, 주문당 평균 판매 수량을 비교하도록 수정
    - 단가 차이를 매출 차이의 원인으로 바로 판단하지 않고 실제 판매 수량을 반영한 가중평균 단가를 계산해 추가로 비교할 것 


### 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [82]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
city_sales = (
    customer_sales_base
    .groupby('city', dropna=False, as_index=False)
    .agg(
        purchasing_customer_count=('customer_id', 'nunique'),
        completed_order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
)
city_comparison = (
    customer_city
    .merge(city_sales, on='city', how='left', validate='one_to_one')
    .fillna({'purchasing_customer_count': 0, 'completed_order_count': 0, 'total_sales': 0})
    .sort_values('total_sales', ascending=False)
)
print('도시별 합계:', city_comparison['total_sales'].sum())
print('completed 원본 합계:', completed_order_sales['line_total'].sum())
print('도시 매칭 누락:', city_comparison['total_sales'].isna().sum())
city_comparison


도시별 합계: 148990000
completed 원본 합계: 148990000
도시 매칭 누락: 0


,city,customer_count,customer_ratio,purchasing_customer_count,completed_order_count,total_sales
0,성남,21,14.00,17,31,24099000
9,고양,11,7.33,9,18,19116000
4,서울,15,10.00,12,29,19097000
5,울산,14,9.33,10,20,15380000
1,광주,17,11.33,11,17,14264000
2,부산,16,10.67,8,13,13604000
6,인천,14,9.33,10,18,12832000
7,대전,14,9.33,8,14,11792000
8,수원,13,8.67,7,14,10620000
3,대구,15,10.00,8,10,8186000


In [84]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
category_product_sales = (
    product_category[['category', 'product_count']]
    .merge(category_sales[['category', 'total_quantity', 'total_sales', 'sales_ratio']],
           on='category', how='left', validate='one_to_one')
    .fillna({'total_quantity': 0, 'total_sales': 0, 'sales_ratio': 0})
)
category_product_sales['product_count_rank'] = category_product_sales['product_count'].rank(
    method='min', ascending=False
).astype(int)
category_product_sales['sales_rank'] = category_product_sales['total_sales'].rank(
    method='min', ascending=False
).astype(int)
category_product_sales = category_product_sales.sort_values('total_sales', ascending=False)
print('카테고리 매칭 누락:', category_product_sales['total_sales'].isna().sum())
print('카테고리 매출 합계:', category_product_sales['total_sales'].sum())
print('상품 수와 매출의 Spearman 상관:', round(
    category_product_sales['product_count'].corr(category_product_sales['total_sales'], method='spearman'), 3
))
category_product_sales


카테고리 매칭 누락: 0
카테고리 매출 합계: 148990000
상품 수와 매출의 Spearman 상관: 0.883


,category,product_count,total_quantity,total_sales,sales_ratio,product_count_rank,sales_rank
0,스포츠,19,295,31743000,21.31,1,1
1,전자기기,17,259,26400000,17.72,2,2
2,생활용품,16,272,23915000,16.05,3,3
3,뷰티,16,223,23383000,15.69,3,4
6,식품,7,133,16573000,11.12,7,5
4,도서,14,149,16389000,11.00,5,6
5,패션,11,111,10587000,7.11,6,7


In [85]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
completed_order_total = (
    order_sales
    .groupby(['order_id', 'payment_method'], dropna=False, as_index=False)
    .agg(order_total=('line_total', 'sum'))
)
payment_order_value = (
    completed_order_total
    .groupby('payment_method', dropna=False, as_index=False)
    .agg(
        completed_order_count=('order_id', 'nunique'),
        total_sales=('order_total', 'sum'),
        avg_order_value=('order_total', 'mean'),
    )
    .sort_values('avg_order_value', ascending=False)
)
payment_order_value['avg_order_value'] = payment_order_value['avg_order_value'].round(0)
print('completed 주문 수:', completed_order_total['order_id'].nunique())
print('결제수단별 합계:', payment_order_value['total_sales'].sum())
print('completed 원본 합계:', completed_order_sales['line_total'].sum())
payment_order_value


completed 주문 수: 184
결제수단별 합계: 148990000
completed 원본 합계: 148990000


,payment_method,completed_order_count,total_sales,avg_order_value
3,naver_pay,51,43500000,852941.0
1,card,39,31806000,815538.0
2,kakao_pay,49,39342000,802898.0
0,bank_transfer,45,34342000,763156.0


In [86]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
monthly_category_sales = (
    sales_items
    .groupby(['order_month', 'category'], dropna=False, as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values(['order_month', 'total_sales'], ascending=[True, False])
)
monthly_category_pivot = monthly_category_sales.pivot(
    index='order_month', columns='category', values='total_sales'
).fillna(0)
monthly_category_pivot['month_total'] = monthly_category_pivot.sum(axis=1)
print('월·카테고리 행 수:', len(monthly_category_sales))
print('월·카테고리 합계:', monthly_category_sales['total_sales'].sum())
print('월별 합계:', monthly_sales['total_sales'].sum())
print('completed 원본 합계:', completed_order_sales['line_total'].sum())
monthly_category_pivot



월·카테고리 행 수: 83
월·카테고리 합계: 148990000
월별 합계: 148990000
completed 원본 합계: 148990000


category,도서,뷰티,생활용품,스포츠,식품,전자기기,패션,month_total
order_month,,,,,,,,
2025-07,656000.0,1336000.0,1117000.0,432000.0,0.0,2328000.0,0.0,5869000.0
2025-08,528000.0,2561000.0,1513000.0,3613000.0,2635000.0,2760000.0,2011000.0,15621000.0
2025-09,844000.0,1573000.0,1250000.0,2815000.0,268000.0,2696000.0,744000.0,10190000.0
2025-10,2550000.0,4606000.0,6027000.0,5799000.0,1899000.0,4118000.0,767000.0,25766000.0
2025-11,870000.0,1317000.0,867000.0,1611000.0,1271000.0,2272000.0,604000.0,8812000.0
2025-12,1975000.0,2305000.0,1060000.0,1103000.0,2130000.0,2928000.0,0.0,11501000.0
2026-01,2352000.0,1497000.0,2138000.0,5966000.0,2083000.0,2192000.0,1195000.0,17423000.0
2026-02,1044000.0,1794000.0,918000.0,2656000.0,1597000.0,464000.0,1276000.0,9749000.0
2026-03,990000.0,1385000.0,2100000.0,3768000.0,2237000.0,1005000.0,1944000.0,13429000.0


In [87]:
# 과제 5. 고객별 order_count와 avg_order_value로 반복 구매와 고액 단발 구매를 구분
# 기준: completed 구매 고객의 median avg_order_value 이상을 고액으로 두고, order_count가 2회 이상이면 반복 구매로 분류
aov_threshold = customer_sales['avg_order_value'].median()
customer_purchase_segment = customer_sales.copy()
customer_purchase_segment['segment'] = np.select(
    [
        customer_purchase_segment['order_count'] >= 2,
        customer_purchase_segment['avg_order_value'] >= aov_threshold,
    ],
    ['repeat_purchase', 'high_value_single_purchase'],
    default='single_purchase',
)
segment_summary = (
    customer_purchase_segment
    .groupby('segment', as_index=False)
    .agg(
        customer_count=('customer_id', 'nunique'),
        total_sales=('total_sales', 'sum'),
        avg_order_count=('order_count', 'mean'),
        avg_order_value=('avg_order_value', 'mean'),
    )
    .sort_values('total_sales', ascending=False)
)
segment_summary['sales_ratio'] = (
    segment_summary['total_sales'] / completed_order_sales['line_total'].sum() * 100
).round(2)
segment_summary[['avg_order_count', 'avg_order_value']] = segment_summary[
    ['avg_order_count', 'avg_order_value']
].round(0)

print('고액 단발 기준(고객별 avg_order_value 중앙값):', aov_threshold)
print('completed 구매 고객 수:', customer_purchase_segment['customer_id'].nunique())
print('세그먼트 고객 수 합계:', segment_summary['customer_count'].sum())
print('세그먼트 매출 합계:', segment_summary['total_sales'].sum())
print('completed 원본 합계:', completed_order_sales['line_total'].sum())
print('검증:', 'PASS' if (
    segment_summary['customer_count'].sum() == customer_purchase_segment['customer_id'].nunique()
    and segment_summary['total_sales'].sum() == completed_order_sales['line_total'].sum()
) else 'CHECK')
segment_summary


고액 단발 기준(고객별 avg_order_value 중앙값): 807000.0
completed 구매 고객 수: 100
세그먼트 고객 수 합계: 100
세그먼트 매출 합계: 148990000
completed 원본 합계: 148990000
검증: PASS


,segment,customer_count,total_sales,avg_order_count,avg_order_value,sales_ratio
1,repeat_purchase,56,111479000,2.0,803493.0,74.82
0,high_value_single_purchase,24,28700000,1.0,1195833.0,19.26
2,single_purchase,20,8811000,1.0,440550.0,5.91


### 과제 6. 프롬프트 작성해보기 

요청: 답할 수 없는 질문을 현재 데이터로 검증가능한 EDA 질문으로 바꿔달라.
 -질문: 어떤 고객이 충성도가 높은가?
 사용 가능한 데이터와 컬럼:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

금액 분석 규칙:
- orders와 연결한 뒤 order_status == "completed"인 주문만 사용합니다.
- total_sales는 completed 주문 상세의 line_total 합계입니다.
- 개인정보나 개별 고객명은 사용하지 않습니다.

조건:
1. 현재 컬럼으로 계산 가능한 질문 후보를 정확히 3개 제안하세요.
2. 각 후보에 필요한 컬럼, 지표, 계산 방법, 검증 방법을 쓰세요.
3. '충성도'를 사실로 단정하지 말고, 관찰 가능한 반복 구매/주문금액/구매 간격으로 표현하세요.
4. 현재 데이터만으로 알 수 없는 점을 각 후보에 명시하세요.
5. 코드는 작성하지 말고, 최종 선택은 분석자가 하도록 남겨 주세요.

### 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


## 5. 다음 분석 우선순위
1. 결제 수단 별 평균 주문 금액이 어떻게 다른가? (평균 주문의 주문당 금액 차이, completed 주문 기준)
2. 도시별 고객 수와 completed 주문 매출이 함께 나타나는가? (고객 기반 규모와 금액이 함께 움직이는지)
3. 카테고리/월별 completed 매출이 어떻게 구성되어있는가? 

### 가장 먼저 확인하고 싶은 이유

1번: 결제수단별 평균 주문금액을 확인하면 매출 차이가 주문 건수 차이에서 나온 것인지, 한 번 주문할 때 쓰는 금액 차이도 함께 있는지 나누어 볼 수 있다. 평균 주문금액은 주문 상세 행 기준이 아니라 주문별 line_total 합계를 먼저 구한 뒤 계산한다. 그래야 한 주문에 여러 상품이 포함된 경우도 제대로 반영할 수 있다.
2번: 고객 규모와 매출 규모가 비슷한 방향으로 나타나는지 보는 기본적인 비교이다. 고객 수가 많아도 실제 구매 고객 수나 주문 횟수, 주문당 금액이 낮으면 매출은 높지 않을 수 있다. 따라서 도시별 총매출만 보지 않고 구매 고객 수와 completed 주문 수도 함께 비교한다. 다만 도시별 차이를 지역 선호나 마케팅 효과로 해석하지는 않아야 한다.
3번: 월과 카테고리를 함께 나누어 매출을 보면 한 카테고리의 비중이 커진 것인지, 여러 카테고리가 함께 늘어난 것인지 확인할 수 있다. 이를 분석하면 추후 어떤 카테고리와 시점을 먼저 볼지 정하는 데 사용할 수 있다. 

### 현재 데이터만으로 단정할 수 없는 것
월별 매출과 카테고리 구성만으로 프로모션, 재고, 계절성, 고객 선호가 매출 변화의 원인이라고 판단할 수는 없다. 또한 2025년 7월과 2026년 7월은 한 달 전체가 포함된 기간이 아니다. 따라서 완전한 월과 같은 기준으로 비교하지 않아야 한다. 이것만으로 결과를 단정할 수 없다.

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 제출합니다.